# CART and Random Forest

**Owner:** Teammate 3

Trees do not need scaled inputs (`kind="plain"`). CART gives one readable tree; Random Forest averages many.

Rules that apply to every notebook: use `prep` and `evaluate` for the split, the preparation and the scoring. Do not write your own. The test set is only touched inside `evaluate.run_feature_steps`.

In [ ]:
# Course      : CS 513 - Data Mining & Machine Learning
# First Name  : <fill in>
# Last Name   : <fill in>
# Id          : <fill in>
# Purpose     : CART and Random Forest on the three feature steps.
#
# AI NOTICE   : The starter code in this notebook was generated by AI (Claude).
#               Prompt and context: docs/AI_PROMPT_LOG.md, entry 1.
#               Mark any cell you add with AI help the same way.

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config
from src import evaluate, plots, prep

df = prep.load_table()
train, test = prep.split_by_building(df)
print(f"{len(df):,} filings, {df['bin'].nunique():,} buildings, "
      f"{df['is_unsafe'].mean():.1%} Unsafe")
print(f"Train: {len(train):,} rows   Test: {len(test):,} rows")

## Train and test on the three feature steps

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree

res_cart, fit_cart = evaluate.run_feature_steps(
    "CART", DecisionTreeClassifier(random_state=config.SEED),
    {"max_depth": [3, 5, 8, 12], "min_samples_leaf": [20, 50, 100],
     "class_weight": [None, "balanced"]},
    kind="plain")

res_rf, fit_rf = evaluate.run_feature_steps(
    "Random Forest", RandomForestClassifier(n_estimators=200, random_state=config.SEED),
    {"max_depth": [8, 12, None], "min_samples_leaf": [5, 20],
     "class_weight": [None, "balanced"]},
    kind="plain")

pd.concat([res_cart, res_rf]).round(3)

## The top of the tree

In [ ]:
best = fit_cart["3_plus_violations"].best_estimator_
names = best.named_steps["prep"].get_feature_names_out()
fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(best.named_steps["model"], max_depth=2, feature_names=names,
          class_names=["Not unsafe", "Unsafe"], filled=False, rounded=True,
          impurity=False, proportion=True, fontsize=9, ax=ax)
plt.show()

## What the Random Forest relies on

In [ ]:
best_rf = fit_rf["3_plus_violations"].best_estimator_
imp = pd.Series(best_rf.named_steps["model"].feature_importances_,
                index=best_rf.named_steps["prep"].get_feature_names_out())
imp = imp.sort_values(ascending=False).head(12)
fig, ax = plots.new_axes("Random Forest: most important inputs",
                         xlabel="Importance", size=(7.5, 4.8))
plots.bars(ax, imp.index, imp.values, color=plots.SERIES[2])
plt.show()

## To do (Teammate 3)

- Read the first three splits out loud. Ask Teammate 1 whether they make engineering sense.
- Did the forest beat the single tree? By how much?
- Save both figures for the deck.